# 深度學習框架應用（大學部）｜主題 13：生成式模型概論（GAN 與 Diffusion）

這本筆記本配合投影片《主題 13：生成式模型概論：GAN 與 Diffusion》。主角是 Fashion-MNIST（10 類衣服鞋子的 28×28 灰階照片），我們只拿 10,000 張訓練、2,000 張驗證，CPU 也跑得完。五題依序練：自編碼器 → VAE → 小 DCGAN → 擴散的加噪排程 → 小網路猜雜訊。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 自編碼器：壓扁再還原 | 第 124 頁 | 第 20～27 頁 |
| 二 | VAE 的重參數化與 KL | 第 125 頁 | 第 32～39 頁 |
| 三 | 小 DCGAN：交替訓練 | 第 126 頁 | 第 41～51、60～86 頁 |
| 四 | 擴散的加噪排程 | 第 127 頁 | 第 93～99 頁 |
| 五 | 小網路猜雜訊 | 第 128 頁 | 第 100～103 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W13_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 由上往下跑：先跑「環境準備」「準備資料」「共用小工具」三格，之後每一題如果有「準備（直接執行）」那一格，也要先跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 有 GPU 會自動用（「執行階段 › 變更執行階段類型」），沒有也沒關係：每一題在 CPU 上幾十秒到幾分鐘就跑完（練習三、五要訓練，比較久）。生成的圖每次訓練都會有一點不同，在 GPU 上跑數字的最後幾位也會不一樣，形狀、參數量、ᾱ_t 這些固定的數字要一樣，訓練的 loss 看大小與趨勢就好。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

下面三格五題都會用到：載入套件、準備 Fashion-MNIST、幾個共用小工具（`count` 數參數、`batches` 裝盒、`show_rows` 畫一排一排的圖）。第一次跑會下載資料，大約 30 秒。

In [ ]:
# ===== 環境準備（直接執行）=====
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch：', torch.__version__, '｜裝置：', dev)

In [ ]:
# ===== 準備資料：Fashion-MNIST（直接執行，不用改）=====
raw = datasets.FashionMNIST('data', train=True, download=True)
X = (raw.data.float() / 255).unsqueeze(1)   # (60000, 1, 28, 28)，像素 0～1
y = raw.targets                              # 0～9：T恤、褲子…短靴
g0 = torch.Generator().manual_seed(42)       # 固定亂數，大家切到同一批
idx = torch.randperm(len(X), generator=g0)
Xtr, ytr = X[idx[:10000]], y[idx[:10000]]    # 訓練用 10,000 張
Xva, yva = X[idx[10000:12000]], y[idx[10000:12000]]  # 驗證用 2,000 張
print('訓練：', tuple(Xtr.shape), '驗證：', tuple(Xva.shape))

In [ ]:
# ===== 共用小工具（直接執行，不用改）=====
NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
         'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

def count(model):
    """模型一共有幾個參數"""
    return sum(p.numel() for p in model.parameters())

def batches(n, bs, seed):
    """把 0～n-1 洗牌後一盒一盒拿出來（湊不滿的最後一盒丟掉）"""
    p = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    for k in range(0, n - bs + 1, bs):
        yield p[k:k + bs]

def show_rows(rows, titles=None, size=1.2):
    """rows：好幾排圖，每一排是 (N, 1, 28, 28)、值在 0～1"""
    n, m = len(rows), len(rows[0])
    fig, ax = plt.subplots(n, m, figsize=(m * size, n * size), squeeze=False)
    for i, r in enumerate(rows):
        for j in range(m):
            ax[i][j].imshow(r[j].detach().cpu().squeeze(), cmap='gray_r', vmin=0, vmax=1)
            ax[i][j].axis('off')
        if titles: ax[i][0].set_title(titles[i], loc='left', fontsize=9)
    plt.tight_layout(); plt.show()

## 練習一：自編碼器：壓扁再還原 (20%)

> 題目在投影片**第 124 頁**｜觀念在第 20～27 頁

**情境**：小智要把每件衣服的 784 個像素塞進只有 32 格的行李箱，到了目的地再還原成一整張圖。行李箱太小，只好學會「衣服最重要的樣子」。

**這題在練**：自己疊出一個堆疊自編碼器（784 → 128 → 32 → 128 → 784），用「輸入就是答案」的方式訓練，再看重建和潛在空間

**步驟**

1. 先執行「練習一：準備」：`fit_ae` 負責訓練（每個 epoch 印驗證集的重建 MSE），`show_recon` 畫重建，`show_latent` 畫潛在空間
2. 在 ✏️ 格填好：潛在層有幾個數、輸出層的激活函數，以及訓練時的「答案」
3. 看三樣東西：潛在碼的形狀與參數量、5 個 epoch 的 MSE、重建圖和潛在空間的散佈圖

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習一：準備（直接執行，不用改）=====
def fit_ae(model, X_in, X_out, epochs=5):
    """輸入 X_in、答案 X_out，用 MSE 練；每個 epoch 印驗證集的重建 MSE"""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(epochs):
        model.train()
        for b in batches(len(X_in), 128, ep):
            loss = F.mse_loss(model(X_in[b].to(dev)), X_out[b].to(dev))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va = F.mse_loss(model(Xva.to(dev)), Xva.to(dev)).item()
        print(f'epoch {ep + 1}  驗證 MSE {va:.4f}')

def show_recon(model, x):
    """上排原圖、下排重建"""
    model.eval()
    with torch.no_grad():
        show_rows([x, model(x.to(dev))], ['input', 'reconstruction'])

def show_latent(enc, X_, y_):
    """把潛在碼用 PCA 降成 2 維，依類別上色（訓練時沒給過類別）"""
    with torch.no_grad():
        c = enc(X_.to(dev)).cpu()
    c = c - c.mean(0)
    _, _, V = torch.pca_lowrank(c, q=2)
    p = c @ V[:, :2]
    for k in range(10):
        m = y_ == k
        plt.scatter(p[m, 0], p[m, 1], s=4, label=NAMES[k])
    plt.legend(markerscale=3, fontsize=7, ncol=2); plt.title('latent (PCA)')
    plt.show()

In [ ]:
# ✏️ 練習一：自編碼器，784 → 32 → 784
torch.manual_seed(42)
enc = nn.Sequential(nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, ___), nn.ReLU())  # ✏️ 壓成 32 個數
dec = nn.Sequential(nn.Linear(32, 128), nn.ReLU(),
    nn.Linear(128, 784), nn.___(),   # ✏️ 壓到 0～1
    nn.Unflatten(1, (1, 28, 28)))
ae = nn.Sequential(enc, dec).to(dev)
print('潛在碼：', tuple(enc(Xva[:4].to(dev)).shape))
print('參數量：', count(ae))
fit_ae(ae, Xtr, ___)                 # ✏️ 答案是誰？
show_recon(ae, Xva[:8])              # 上排原圖、下排重建
show_latent(enc, Xva, yva)           # 潛在空間

#### 📸 截圖 ①

跑完上面的格子，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「潛在碼」「參數量」兩行、5 行驗證 MSE、重建圖（上排原圖、下排重建）和潛在空間散佈圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 印出來的參數量是多少？請寫出四層各是怎麼算的（每一層都是「輸入×輸出＋輸出」）。**

（在這裡寫）

**2. 訓練時為什麼把 Xtr 同時當輸入和答案？看潛在空間的散佈圖，哪幾類衣服自己聚成一團？訓練時有沒有給它類別？**

（在這裡寫）

## 練習二：VAE 的重參數化與 KL (20%)

> 題目在投影片**第 125 頁**｜觀念在第 32～39 頁

**情境**：一般自編碼器的潛在空間有洞、又不整齊，隨手抽一點常畫出四不像。VAE 讓每張圖佔一小塊區域，而且全部往原點靠，訓練完就能從 N(0, I) 直接抽。

**這題在練**：先用一組數字手算重參數化和 KL，再訓練一個潛在空間只有 2 維的 VAE，從 N(0, I) 抽樣並畫出潛在地圖

**步驟**

1. 先執行「練習二：準備」：`VAE` 類別（編碼器輸出 μ 和 log σ²）、`fit_vae` 訓練、`show_samples` 抽樣、`show_map` 畫潛在地圖
2. 在 ✏️ 格填好三個空格：σ 怎麼從 log σ² 算、重參數化的 z、KL 裡的 σ²
3. 對答案：σ、z、KL 三個數字；再看 10 個 epoch 的訓練、16 張抽樣圖和 8×8 的潛在地圖

> 這一題要訓練 10 個 epoch，CPU 上大約 1 分鐘。

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行，不用改，訓練約 1 分鐘）=====
class VAE(nn.Module):
    """潛在空間 d 維的 VAE：784 → 256 → (μ, log σ²) → z → 256 → 784"""
    def __init__(self, d=2):
        super().__init__()
        torch.manual_seed(42)                 # 每次建立都從同一組亂數開始
        self.d = d
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                                 nn.Linear(256, 2 * d))      # 輸出 μ 和 log σ²
        self.dec = nn.Sequential(nn.Linear(d, 256), nn.ReLU(),
                                 nn.Linear(256, 784), nn.Sigmoid(),
                                 nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        mu, logvar = self.enc(x).chunk(2, dim=-1)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)   # 重參數化
        return self.dec(z), mu, logvar

def fit_vae(vae, epochs=10):
    """損失＝重建誤差（BCE 加總）＋ KL；每個 epoch 印最後一盒的兩項"""
    opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
    for ep in range(epochs):
        for b in batches(len(Xtr), 128, ep):
            x = Xtr[b].to(dev)
            out, mu, logvar = vae(x)
            rec = F.binary_cross_entropy(out, x, reduction='sum') / len(x)
            kl = (-0.5 * torch.sum(1 + logvar - mu**2 - logvar.exp(), -1)).mean()
            opt.zero_grad(); (rec + kl).backward(); opt.step()
        print(f'epoch {ep + 1}  重建 {rec.item():.1f}  KL {kl.item():.2f}')

def show_samples(vae, n=16):
    """從 N(0, I) 抽 n 個 z，交給解碼器畫"""
    torch.manual_seed(7)
    with torch.no_grad():
        imgs = vae.dec(torch.randn(n, vae.d, device=dev))
    show_rows([imgs[:8], imgs[8:]], ['samples from N(0, I)', ''])

def show_map(vae, k=8):
    """2 維潛在空間的地圖：z 從 -2 走到 2，每一格解碼成一張圖"""
    v = torch.linspace(-2, 2, k)
    with torch.no_grad():
        rows = [vae.dec(torch.stack([torch.tensor([a, b]) for a in v]).to(dev))
                for b in v.flip(0)]
    show_rows(rows, size=0.9)

In [ ]:
# ✏️ 練習二：VAE 的重參數化和 KL（先手算一次）
mu = torch.tensor([0.5, -1.0])         # 編碼器給的平均
logvar = torch.tensor([0.0, math.log(0.25)])  # log σ²
std = torch.exp(___ * logvar)          # ✏️ σ
eps = torch.tensor([1.0, -2.0])        # 抽到的 ε
z = mu + std * ___                     # ✏️ 重參數化
kl = -0.5 * torch.sum(1 + logvar - mu**2
                      - logvar.___())  # ✏️ σ²
print('σ =', std, ' z =', z)
print('KL =', round(kl.item(), 4))
vae = VAE(d=2).to(dev)                 # 準備格寫好的
fit_vae(vae, epochs=10)                # 約 1 分鐘
show_samples(vae)                      # 從 N(0, I) 抽 z
show_map(vae)                          # 潛在地圖

#### 📸 截圖 ②

跑完上面的格子，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：σ、z、KL 兩行、10 行訓練紀錄、16 張抽樣圖和潛在地圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 寫出 σ、z、KL 的計算過程（μ＝[0.5, −1]、log σ²＝[0, ln 0.25]、ε＝[1, −2]）。**

（在這裡寫）

**2. 為什麼要用 z＝μ＋σ·ε 這種寫法，不直接「從 N(μ, σ²) 抽一個」？看潛在地圖，從左上走到右下，衣服怎麼變？**

（在這裡寫）

## 練習三：小 DCGAN：交替訓練 (20%)

> 題目在投影片**第 126 頁**｜觀念在第 41～51、60～86 頁

**情境**：做假鈔的人（生成器 G）和驗鈔員（判別器 D）輪流練：D 學著分真假，G 學著騙過 D。這次的 DCGAN 比投影片的範例小一點（通道數 64、32），CPU 也跑得動。

**這題在練**：親手寫出 GAN 最關鍵的兩行：更新 D 時把假圖 detach、更新 G 時把假圖標成 1（非飽和技巧），再看格線圖和 loss 曲線

**步驟**

1. 先執行「練習三：準備」：會建好生成器 g、判別器 d、兩個 Adam（lr＝2e-4、betas＝(0.5, 0.999)），並印出兩個網路的輸出形狀和參數量
2. 再執行「練習三：小工具」，然後在 ✏️ 格填好兩個空格，訓練 5 個 epoch
3. 看第 1 和第 5 個 epoch 的格線圖（同一組 fixed_z），以及 D、G 兩條 loss 曲線

> 這一題要訓練 5 個 epoch，CPU 上大約 3～8 分鐘（看電腦），有 GPU 不到 1 分鐘。訓練中途可以先讀投影片「該觀察什麼」那頁。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改；會印出兩個網路的形狀和參數量）=====
Z = 100                                  # 雜訊長度
Xg = Xtr * 2 - 1                         # 換到 [-1, 1]，對上生成器最後的 Tanh
torch.manual_seed(42)
g = nn.Sequential(                       # 生成器：1×1 → 7×7 → 14×14 → 28×28
    nn.ConvTranspose2d(Z, 64, 7, 1, 0, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
    nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
    nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh()).to(dev)
d = nn.Sequential(                       # 判別器：28×28 → 14×14 → 7×7 → 1 個 logit
    nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2, True),
    nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),
    nn.Conv2d(64, 1, 7, 1, 0), nn.Flatten()).to(dev)

def init_w(m):                           # DCGAN 論文建議的初始化
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
g.apply(init_w); d.apply(init_w)
opt_g = torch.optim.Adam(g.parameters(), 2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(d.parameters(), 2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
ones = lambda n: torch.ones(n, 1, device=dev)     # n 個標籤 1
zeros = lambda n: torch.zeros(n, 1, device=dev)   # n 個標籤 0
fixed_z = torch.randn(32, Z, 1, 1, generator=torch.Generator().manual_seed(7)).to(dev)
log = {'D': [], 'G': []}                 # 每一步的兩個 loss
with torch.no_grad():
    print('G 的輸出：', tuple(g(fixed_z[:4]).shape), '｜參數量：', count(g))
    print('D 的輸出：', tuple(d(Xg[:4].to(dev)).shape), '｜參數量：', count(d))

def show_grid(title):
    """用同一組 fixed_z 產 32 張圖（產完切回 train）"""
    g.eval()
    with torch.no_grad():
        imgs = (g(fixed_z) + 1) / 2      # 從 [-1, 1] 換回 [0, 1]
    g.train()
    show_rows([imgs[k:k + 8] for k in range(0, 32, 8)], [title, '', '', ''], size=0.9)

def show_losses(log):
    plt.figure(figsize=(8, 3))
    plt.plot(log['D'], label='D loss', lw=1); plt.plot(log['G'], label='G loss', lw=1)
    plt.axhline(1.386, ls='--', c='gray', lw=1); plt.axhline(0.693, ls=':', c='gray', lw=1)
    plt.xlabel('step'); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# ===== 練習三：小工具（直接執行，不用改）=====
def report3(ep, loss_d, loss_g):
    """每個 epoch 結束：印最後一盒的 D、G loss；第 1 和第 5 個 epoch 存格線圖、最後畫 loss 曲線"""
    print(f'epoch {ep}  D {loss_d.item():.3f}  G {loss_g.item():.3f}'
          f'  （D 給假圖約 {math.exp(-loss_g.item()):.2f} 分）')
    if ep in (1, 5): show_grid(f'epoch {ep}')
    if ep == 5: show_losses(log)

In [ ]:
# ✏️ 練習三：小 DCGAN，交替訓練 5 個 epoch
for ep in range(1, 6):
    for b in batches(len(Xg), 128, ep):
        real = Xg[b].to(dev); B = len(real)
        fake = g(torch.randn(B, Z, 1, 1, device=dev))
        loss_d = (bce(d(real), ones(B)) +
            bce(d(fake.___()), zeros(B)))     # ✏️ 切斷
        opt_d.zero_grad(); loss_d.backward()
        opt_d.step()                          # 只改 D
        loss_g = bce(d(fake), ___(B))         # ✏️ 標成？
        opt_g.zero_grad(); loss_g.backward()
        opt_g.step(); log['D'] += [loss_d.item()]
        log['G'] += [loss_g.item()]
    report3(ep, loss_d, loss_g)  # 印 loss、存格線圖

#### 📸 截圖 ③

跑完上面的格子，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：G、D 的形狀與參數量兩行、5 行 epoch 紀錄、第 1 和第 5 個 epoch 的格線圖、loss 曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 更新 D 的那一行為什麼要 fake.detach()？更新 G 時為什麼把假圖標成 1（不是 0）？**

（在這裡寫）

**2. 比較第 1 和第 5 個 epoch 的格線圖，用自己的話描述差別。用最後一個 epoch 的 G loss 換算 D 給假圖幾分（e 的負 G loss 次方），這算健康的拉扯嗎？**

（在這裡寫）

## 練習四：擴散的加噪排程 (20%)

> 題目在投影片**第 127 頁**｜觀念在第 93～99 頁

**情境**：擴散模型先學「把圖弄髒」：DDPM 原版用 1000 步，每一步的 β 從 0.0001 線性加到 0.02。ᾱ_t 是原圖還剩多少，一路乘下來。

**這題在練**：自己算出線性排程的 ᾱ_t，寫出「一步跳到第 t 步」的公式，並看同一張圖在不同 t 的樣子

**步驟**

1. 先執行「練習四：準備」：`report4` 印 5 個 t 的 ᾱ_t 和兩個係數，`show_noisy` 畫同一張圖在不同 t 的樣子，`show_schedule` 畫係數曲線
2. 在 ✏️ 格填好最後一步的 β、累乘的函式，以及 jump 裡雜訊係數的開根號
3. 和投影片「ᾱ_t 走 1000 步」那張表對答案，再看圖：t 越大，衣服越看不見

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改）=====
x0 = Xva[:1] * 2 - 1                     # 一張圖，換到 [-1, 1]

def report4(abar):
    for t in (1, 100, 250, 500, 1000):
        a = abar[t - 1].item()
        print(f't={t:4d}  ᾱ={a:.5f}  原圖×{a ** 0.5:.3f}  雜訊×{(1 - a) ** 0.5:.3f}')
    print('雜訊開始比原圖多：t =', int((abar < 0.5).nonzero()[0]) + 1)

def show_noisy(jump, ts):
    """同一張圖、同一把雜訊，在不同 t 的樣子"""
    eps = torch.randn(x0.shape, generator=torch.Generator().manual_seed(3))
    imgs = torch.cat([((jump(x0, t, eps) + 1) / 2).clamp(0, 1) for t in ts])
    show_rows([imgs], ['t = ' + ' / '.join(map(str, ts))], size=1.6)

def show_schedule(abar):
    """線性排程的 √ᾱ_t（原圖）、√(1−ᾱ_t)（雜訊），再畫餘弦排程比一比"""
    t = torch.arange(1, 1001); s = 0.008
    f = lambda u: torch.cos((u / 1000 + s) / (1 + s) * math.pi / 2) ** 2
    plt.plot(t, abar.sqrt(), label='sqrt(abar), linear')
    plt.plot(t, (1 - abar).sqrt(), label='sqrt(1 - abar), linear')
    plt.plot(t, f(t) / f(torch.tensor(0.)), '--', label='abar, cosine')
    plt.xlabel('t'); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# ✏️ 練習四：線性排程，ᾱ_t 一路乘下來
T = 1000
betas = torch.linspace(1e-4, ___, T)   # ✏️ 最後一步
alphas = 1 - betas
abar = torch.___(alphas, dim=0)        # ✏️ 一路乘
report4(abar)                          # 印 5 個 t
def jump(x0, t, eps):                  # 一步跳到第 t 步
    a = abar[t - 1].view(-1, 1, 1, 1).to(x0.device)
    return a.sqrt() * x0 + (1 - a).___() * eps  # ✏️
show_noisy(jump, [1, 100, 250, 500, 1000])
show_schedule(abar)                    # 原圖和雜訊的係數

#### 📸 截圖 ④

跑完上面的格子，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：5 行 ᾱ_t、「雜訊開始比原圖多」那一行、同一張圖在 5 個 t 的樣子和係數曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. t＝250 和 t＝500 時，原圖係數和雜訊係數各是多少？雜訊從第幾步開始比原圖多？**

（在這裡寫）

**2. 為什麼可以「一步跳到第 t 步」，不用真的加 t 次雜訊？t＝1000 時圖幾乎只剩雜訊，這對生成有什麼好處？**

（在這裡寫）

## 練習五：小網路猜雜訊 (20%)

> 題目在投影片**第 128 頁**｜觀念在第 100～103 頁

**情境**：擴散模型唯一要學的事：看一張帶噪的圖 x_t 和步數 t，猜出當初加進去的雜訊 ε。猜到雜訊，就能把原圖估回來。

**這題在練**：寫出擴散模型的訓練目標（猜雜訊的 MSE），訓練一個很小的 U-Net 600 步，再用 x̂₀＝(x_t − √(1−ᾱ_t)·ε̂) ÷ √ᾱ_t 把圖估回來

**步驟**

1. 先執行「練習五：準備」：`EpsNet` 是很小的 U-Net（28 → 14 → 28、一條跳接、t 當成第 2 個通道），`pick` 抽乾淨的圖，`show_x0` 畫結果。要先做完練習四（會用到 `abar` 和 `jump`）
2. 在 ✏️ 格填好兩個空格：MSE 要和誰比、估回原圖時要除以什麼，跑 600 步
3. 看最後的圖和三行 MSE：比較 t＝50／200／500 時，帶噪的圖和估回來的圖各差原圖多少

> 這一題要訓練 600 步，CPU 上大約 2～6 分鐘（看電腦），有 GPU 不到 1 分鐘。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備（直接執行，不用改；要先做完練習四）=====
class EpsNet(nn.Module):
    """很小的 U-Net：28 → 14 → 28、一條跳接；t 變成一張「時間圖」當第 2 個通道"""
    def __init__(self, c=32):
        super().__init__()
        torch.manual_seed(0)
        self.down = nn.Sequential(nn.Conv2d(2, c, 3, 1, 1), nn.ReLU(),
                                  nn.Conv2d(c, c, 4, 2, 1), nn.ReLU())   # 28 → 14
        self.mid = nn.Sequential(nn.Conv2d(c, c, 3, 1, 1), nn.ReLU())
        self.up = nn.Sequential(nn.ConvTranspose2d(c, c, 4, 2, 1), nn.ReLU())  # 14 → 28
        self.out = nn.Conv2d(c + 2, 1, 3, 1, 1)                       # 跳接：把輸入接回來
    def forward(self, xt, t):
        tt = (t.float().to(xt.device) / 1000).view(-1, 1, 1, 1).expand_as(xt)
        h = torch.cat([xt, tt], 1)
        return self.out(torch.cat([self.up(self.mid(self.down(h))), h], 1))

X0 = Xtr * 2 - 1                         # 訓練圖換到 [-1, 1]
def pick(n):
    """隨機抽 n 張乾淨的圖"""
    return X0[torch.randint(0, len(X0), (n,))].to(dev)

def show_x0(net, x0_hat, ts=(50, 200, 500)):
    """每個 t：上排帶噪的 x_t、下排網路估回的 x̂_0；印出兩者和原圖差多少（MSE）"""
    x = Xva[:6] * 2 - 1
    rows, titles = [(x + 1) / 2], ['original']
    net.eval()
    for t in ts:
        eps = torch.randn(x.shape, generator=torch.Generator().manual_seed(t))
        tt = torch.full((6,), t)
        xt = jump(x.to(dev), tt, eps.to(dev))
        with torch.no_grad():
            xh = x0_hat(xt, tt, net(xt, tt)).clamp(-1, 1)
        m1, m2 = F.mse_loss(xt.cpu(), x).item(), F.mse_loss(xh.cpu(), x).item()
        print(f't={t:3d}  帶噪的 MSE {m1:.3f}  →  估回來的 MSE {m2:.3f}')
        rows += [((xt + 1) / 2).clamp(0, 1), (xh + 1) / 2]
        titles += [f'x_t (t={t})', 'estimated x_0']
    net.train()
    show_rows(rows, titles, size=1.1)

In [ ]:
# ✏️ 練習五：小網路學「猜雜訊」，再把圖估回來
net = EpsNet().to(dev)               # 準備格的小 U-Net
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for step in range(1, 601):
    x0, t = pick(64), torch.randint(1, 1001, (64,))
    eps = torch.randn_like(x0)       # 要加的雜訊
    pred = net(jump(x0, t, eps), t)  # 網路猜的雜訊
    loss = F.mse_loss(pred, ___)     # ✏️ 答案是？
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: print(step, round(loss.item(), 4))
def x0_hat(xt, t, e):                # 猜到雜訊→算回原圖
    a = abar[t - 1].view(-1, 1, 1, 1).to(xt.device)
    return (xt - (1 - a).sqrt() * e) / a.___()  # ✏️
show_x0(net, x0_hat)                 # t＝50／200／500

#### 📸 截圖 ⑤

跑完上面的格子，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：每 100 步的 loss、t＝50／200／500 三行 MSE，以及原圖／帶噪／估回的圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 訓練時 loss 是怎麼變的？為什麼目標是「猜雜訊 ε」而不直接猜原圖？**

（在這裡寫）

**2. 三個 t 的「估回來的 MSE」都比「帶噪的 MSE」小嗎？t 越大，估回來的圖有什麼變化？為什麼？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：為什麼 GAN 更新 D 時要 detach） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W13_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。